# RP-OD on COCO 2017: pretraining and benchmark

One COCO pretraining per model size (`s`, `m`, `l`), from ImageNet backbones. Each one serves twice:

* **pretrained weights** for training that size on our own datasets;
* **COCO numbers** comparable with the literature: AP on val2017 with pycocotools, 640×640, 300 queries, up to 100 detections.

Steps: 1 data · 2 augmentations · 3 dataset check · 4 the sizes · 5 pretraining · 6 follow a run · 7 official numbers.

Long jobs run **detached**: closing the browser or restarting this kernel does not stop them. Their output goes to `/work/logs/<name>.log`, and the training curves to `/work/runs/train/<name>/results.csv`. Every run also saves its whole training state after each epoch, so an interrupted run can be resumed (cell after step 6).

In [ ]:
import os, signal, subprocess, sys, json, csv
from pathlib import Path

WORK = Path('/work')        # where the training container mounts the data: change it to yours
REPO = WORK / 'RP-OD'
BENCH = REPO / 'benchmarks' / 'coco'
COCO = WORK / 'coco'
LOGS = WORK / 'logs'
RUNS = WORK / 'runs' / 'train'
sys.path[:0] = [str(REPO), str(BENCH)]
os.chdir(WORK)
LOGS.mkdir(exist_ok=True)

import torch
print('torch', torch.__version__, '·', torch.cuda.get_device_name(0))


def launch(name, *commands, workers=16):
    """Run one or more python commands one after the other, detached, logging to logs/<name>.log."""
    chain = ' && '.join(' '.join([sys.executable, '-u', *map(str, c)]) for c in commands)
    log = open(LOGS / f'{name}.log', 'a')
    p = subprocess.Popen(['bash', '-c', chain], cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                         env={**os.environ, 'RPOD_WORKERS': str(workers)})
    (LOGS / f'{name}.pid').write_text(str(p.pid))
    print(f'{name}: started (pid {p.pid}) · log {LOGS / name}.log')


def running(name):
    pid_file = LOGS / f'{name}.pid'
    if not pid_file.exists():
        return False
    try:
        os.kill(int(pid_file.read_text()), 0)
        return True
    except ProcessLookupError:
        return False


def stop(name):
    os.killpg(int((LOGS / f'{name}.pid').read_text()), signal.SIGTERM)


def tail(name, n=8):
    lines = (LOGS / f'{name}.log').read_text().splitlines()
    print(f"[{'running' if running(name) else 'finished'}]", *lines[-n:], sep='\n')

## 1. Data

Downloads COCO 2017 (~19 GB), extracts it and writes the labels in the format RP-OD reads (`coco/coco.yaml`). It takes a while; re-run the cell to see how it is going. If it is interrupted, launching it again resumes the download.

In [ ]:
if (COCO / 'coco.yaml').exists():
    print('COCO is ready:', COCO / 'coco.yaml')
elif not running('prepare'):
    launch('prepare', [BENCH / 'prepare_coco.py', '--root', COCO])
else:
    tail('prepare')

## 2. The augmentations, on real images

RP-OD's default augmentation (`rpod.data.augment`, what `RPOD.train` uses unless told otherwise) is the RT-DETR / D-FINE recipe: photometric distortion, zoom-out, IoU crop and horizontal flip, plus mosaic (half of the images) in the first quarter of the epochs; nothing in the last 4 %. Each row: the original and four augmented versions, with their boxes.

In [ ]:
import numpy as np
from types import SimpleNamespace
from IPython.display import Image as Show
from rpod.data.dataset import Sample
from rpod.data.sources import read_image
from rpod.utils.plots import plot_image_grid
from rpod.data.augment import default_augment as augment

names = dict(enumerate(json.loads((COCO / 'classes.json').read_text())['names']))
items = []
for path in sorted((COCO / 'images' / 'train2017').glob('*.jpg'))[:4]:
    image = read_image(path)
    h, w = image.shape[:2]
    lab = np.loadtxt(COCO / 'labels' / 'train2017' / (path.stem + '.txt'), ndmin=2)
    b = lab[:, 1:] * [w, h, w, h]
    sample = Sample(image, np.c_[b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2].astype(np.float32), lab[:, 0].astype(int))
    for k in range(5):
        ctx = SimpleNamespace(epoch=0, epochs=120, rng=np.random.default_rng(k), imgsz=[640, 640],
                              sample=lambda n: [sample] * n)          # mosaic of the same image, to show it
        s = sample if k == 0 else augment(sample, ctx)
        items.append((s.image, np.zeros((0, 4)), np.zeros(0), s.boxes, np.ones(len(s.boxes)), s.labels))
plot_image_grid(items, names, 0.5, LOGS / 'augmentations.jpg')
Show(filename=str(LOGS / 'augmentations.jpg'), width=1000)

## 3. Dataset check

Reads every image and label once (~15 min the first time, then cached): unreadable images, wrong boxes, duplicates.

In [ ]:
from rpod.data import check_dataset
ds = check_dataset(COCO / 'coco.yaml')
print(json.dumps(ds['stats'], indent=1))
print(len(ds['warnings']), 'warnings')

## 4. The sizes

Parameters per block and GFLOPs of each size at 640×640.

In [ ]:
from rpod import RPOD
for size in ('s', 'm', 'l'):
    RPOD(size).info(imgsz=640)
    print()

## 5. COCO pretraining of a size

120 epochs, batch 32, multi-scale (height and width drawn separately: 320–960 in the first quarter, 480–800 until the last sixth, then 640), learning rate 2e-4 flat for the first half and decaying over the second; backbone learning rate 1e-4 (`s`), 2e-5 (`m`), 1e-5 (`l`). About two days for `s` on the 4090.

In [ ]:
SIZE = 's'
name = f'rpod_{SIZE}'
if not running(name):
    launch(name, [BENCH / 'train_coco.py', '--name', name, '--size', SIZE])
else:
    tail(name)

## 6. Follow the runs

Validation (val2017, RP-OD evaluator) after every epoch. Re-run to refresh.

In [ ]:
import matplotlib.pyplot as plt

def curves(*names):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for name in names:
        f = RUNS / name / 'results.csv'
        if not f.exists():
            continue
        rows = list(csv.DictReader(open(f)))
        ep = [int(r['epoch']) for r in rows]
        for ax, key in zip(axes, ('loss', 'map50_95', 'map50')):
            ax.plot(ep, [float(r[key]) for r in rows], label=name)
            ax.set_title(key); ax.set_xlabel('epoch'); ax.grid(alpha=.3)
        last = rows[-1]
        print(f"{name}: epoch {last['epoch']} · mAP50-95 {float(last['map50_95']):.3f} · mAP50 {float(last['map50']):.3f} "
              f"· {float(last['seconds']) / 60:.0f} min/epoch")
    axes[0].legend()
    plt.show()

curves('rpod_s', 'rpod_m', 'rpod_l')

If a run was interrupted (machine restarted, container stopped...), it continues where it stopped, in the same folder:

In [ ]:
def resume(name):
    launch(f'{name}_resume', [BENCH / 'train_coco.py', '--resume', RUNS / name / 'weights' / 'state.pt'])

# resume('rpod_s')

## 7. Official COCO numbers

pycocotools on `instances_val2017.json`, exactly as the papers, for every size whose pretraining has finished.

In [ ]:
from eval_coco import evaluate as coco_eval

def table(rows):
    keys = ['model', 'params_M', 'AP', 'AP50', 'AP75', 'APs', 'APm', 'APl']
    print(''.join(f'{k:>10}' if k != 'model' else f'{k:<40}' for k in keys))
    for r in rows:
        print(''.join(f'{r[k]:>10}' if k != 'model' else f'{r[k]:<40}' for k in keys))

table([coco_eval(RUNS / f'rpod_{size}' / 'weights' / 'best.pt') for size in ('s', 'm', 'l')
       if (RUNS / f'rpod_{size}' / 'weights' / 'best.pt').exists()])